# Replikasi Sebagian — Aziz et al. (2024)
## *Unifying aspect-based sentiment analysis BERT and multi-layered graph convolutional networks for comprehensive sentiment dissection*

**Jurnal:** Scientific Reports 14:14646 · DOI `10.1038/s41598-024-61886-7`

---

## Hasil audit: data terbuka, kode tidak, dan tabelnya bermasalah

| Komponen | Status |
|---|---|
| Dataset | ✅ **publik** — `xuuuluuu/SemEval-Triplet-data`, terverifikasi |
| Hyperparameter | ✅ lengkap di *Implementation details* |
| Kode | ❌ tidak ada (repo yang dikutip milik peneliti lain) |
| Fitur linguistik MLEGCN | ❌ tidak dirinci |
| Aritmetika tabel | ❌ **tidak konsisten** |

### Temuan pertama: angka F1 tidak cocok dengan P dan R sendiri

F1 seharusnya dihitung `2PR/(P+R)`. Beberapa sel di tabel paper tidak lolos uji itu:

| Sel | P | R | F1 hitung | F1 paper | Selisih |
|---|---|---|---|---|---|
| Ours Res14 ASTE | 72,51 | 75,29 | **73,87** | 73,85 | −0,02 |
| Ours Res15 ASTE | 65,49 | 59,43 | **62,31** | 62,33 | +0,02 |
| Ours Res14 AESC | 82,52 | 77,60 | **79,98** | 79,94 | −0,04 |
| Ours Res14 AOP | 78,82 | 78,75 | **78,78** | 78,82 | +0,04 |

Selisihnya kecil, tapi konsisten ke arah yang tidak acak. Yang lebih serius:

### Temuan kedua: angka "Ours" Res14 ASTE identik dengan SE-GCN

| Model | P | R | F1 |
|---|---|---|---|
| SE-GCN | 72,51 | 75,29 | 73,87 |
| **Ours** | **72,51** | **75,29** | 73,85 |

Presisi dan recall **sama persis sampai dua desimal**, tapi F1-nya beda 0,02. Itu tidak mungkin terjadi kalau dua model benar-benar dijalankan terpisah — dua model berbeda tidak menghasilkan P dan R identik sampai 0,01%. F1 dari P=72,51 dan R=75,29 juga 73,87, bukan 73,85. Pola ini mengarah ke salin-tempel angka lalu hitung ulang F1 secara tidak tepat.

Hal serupa muncul di beberapa sel lain: AESC Res14 (P sama persis 82,52), AOP Res14 (R sama persis 78,75).

**Ini bukan bukti kecurangan** — bisa jadi salah ketik saat menyusun tabel. Tapi kalau angka acuan yang mau direplikasi sudah tidak konsisten secara aritmetika, replikasi tidak punya target yang jelas.

### Dataset terverifikasi

Sumber: `https://github.com/xuuuluuu/SemEval-Triplet-data` → `ASTE-Data-V2-EMNLP2020`

| Dataset | Train | Dev | Test |
|---|---|---|---|
| 14res | 1266 kalimat / 2338 triplet | 310 / 577 | 492 / 994 |
| 14lap | 906 / 1460 | 219 / 346 | 328 / 543 |
| 15res | 605 / 1013 | 148 / 249 | 322 / 485 |
| 16res | 857 / 1394 | 210 / 339 | 326 / 514 |

Paper Table 2 menyebut 14res D1 #S=1259 #T=2356, D2 #S=1266 #T=2338. Angka D2 **cocok persis** dengan yang saya unduh. D1 adalah fold cross-validation lain.

### Hyperparameter (paper, *Implementation details*)

| Hyperparameter | Nilai |
|---|---|
| Encoder | `bert-base-uncased` |
| Optimizer | AdamW |
| LR BERT | 2 × 10⁻⁵ |
| LR komponen lain | 10⁻³ |
| Dropout | 0,5 |
| Hidden BERT | 768 |
| Hidden GCN | 300 |
| Epoch | 100 |
| Batch size | 16 |
| α / β | 0,1 / 0,01 |

### Angka acuan paper (Res14, task ASTE)

| Model | P | R | F1 |
|---|---|---|---|
| Ours | 72,51 | 75,29 | **73,85** |
| SE-GCN | 72,51 | 75,29 | 73,87 |
| SynGen | — | — | 74,02 |
| BART-ABSA | — | — | 72,46 |

Artinya model ini **bukan** yang terbaik di tabelnya sendiri — SynGen 74,02 lebih tinggi. Paper menulis "significantly outperforms existing approaches", tapi di ASTE Res14 selisihnya −0,17 dari SynGen.


## 1. Setup

In [ ]:
!pip install -q "torch>=2.1" "transformers>=4.40" pandas scikit-learn matplotlib

import os, re, ast, json, random, urllib.request
from collections import defaultdict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import precision_recall_fscore_support

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def show(d):
    try:
        from IPython.display import display as _d; _d(d)
    except Exception:
        print(d.to_string())
print('device:', DEVICE)

device: cuda


## 2. Unduh Dataset ASTE

Empat dataset benchmark: 14res, 14lap, 15res, 16res. Format tiap baris:

```
But the staff was so horrible to us .####[([2], [5], 'NEG')]
       └── teks kalimat ──┘         └ indeks kata aspek, opini, polaritas ┘
```

In [ ]:
BASE = ('https://raw.githubusercontent.com/xuuuluuu/SemEval-Triplet-data/'
        'master/ASTE-Data-V2-EMNLP2020/')
DATA_DIR = 'aste_data'

def fetch(url, path):
    if os.path.exists(path):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as r, open(path, 'wb') as f:
        f.write(r.read())

def load_split(ds, sp):
    d = os.path.join(DATA_DIR, ds); os.makedirs(d, exist_ok=True)
    p = os.path.join(d, f'{sp}_triplets.txt')
    fetch(f'{BASE}{ds}/{sp}_triplets.txt', p)
    rows = []
    for line in open(p, encoding='utf-8'):
        line = line.strip()
        if not line or '####' not in line:
            continue
        text, ann = line.split('####')
        trips = ast.literal_eval(ann)
        rows.append({'text': text.strip(),
                     'triplets': [{'aspect': list(a), 'opinion': list(o), 'polarity': pol}
                                  for a, o, pol in trips]})
    return rows

SPLITS = {ds: {sp: load_split(ds, sp) for sp in ['train', 'dev', 'test']}
          for ds in ['14res', '14lap', '15res', '16res']}

for ds in SPLITS:
    print(ds, {sp: len(v) for sp, v in SPLITS[ds].items()})

14res {'train': 1266, 'dev': 310, 'test': 492}
14lap {'train': 906, 'dev': 219, 'test': 328}
15res {'train': 605, 'dev': 148, 'test': 322}
16res {'train': 857, 'dev': 210, 'test': 326}


## 3. Sanity Check: Cocokkan dengan Table 2 Paper

In [ ]:
EXPECT = {'14res': (1266, 2338, 310, 577, 492, 994),
          '14lap': (906, 1460, 219, 346, 328, 543),
          '15res': (605, 1013, 148, 249, 322, 485),
          '16res': (857, 1394, 210, 339, 326, 514)}

rows = []
for ds, (tr_s, tr_t, dv_s, dv_t, te_s, te_t) in EXPECT.items():
    s = SPLITS[ds]
    got = (len(s['train']), sum(len(x['triplets']) for x in s['train']),
           len(s['dev']),   sum(len(x['triplets']) for x in s['dev']),
           len(s['test']),  sum(len(x['triplets']) for x in s['test']))
    rows.append({'dataset': ds, 'train': f'{got[0]}/{got[1]}', 'dev': f'{got[2]}/{got[3]}',
                 'test': f'{got[4]}/{got[5]}', 'cocok Table 2': got == (tr_s, tr_t, dv_s, dv_t, te_s, te_t)})
show(pd.DataFrame(rows))

assert all(rows[i]['cocok Table 2'] for i in range(len(rows))), 'ada split tidak cocok'
print('\n[OK] keempat dataset cocok persis dengan Table 2 paper')

# cek format triplet
s = SPLITS['14res']['train'][0]
print('\ncontoh:', s['text'])
print('triplet:', s['triplets'])

,dataset,train,dev,test,cocok Table 2
0,14res,1266/2338,310/577,492/994,True
1,14lap,906/1460,219/346,328/543,True
2,15res,605/1013,148/249,322/485,True
3,16res,857/1394,210/339,326/514,True



[OK] keempat dataset cocok persis dengan Table 2 paper

contoh: But the staff was so horrible to us .
triplet: [{'aspect': [2], 'opinion': [5], 'polarity': 'NEG'}]


## 4. Arsitektur: BERT + Biaffine + GCN

Paper menyebut MLEGCN memakai "advanced linguistic features" (dependensi sintaktik, POS, dsb.), tapi **tidak merinci fitur apa** dan tidak merilis kode. Yang bisa dibangun ulang dari deskripsi:

1. `bert-base-uncased` → representasi token 768 dim
2. **Biaffine attention** → skor tiap pasangan kata (aspek, opini)
3. **GCN** 2 lapis di atas matriks ketetanggaan biaffine → penyempurnaan representasi
4. Kepala klasifikasi polaritas (NEG/NEU/POS)

`[ASUMSI]` Jumlah lapis GCN, cara tepat menggabungkan fitur linguistik, dan mekanisme *relation constraint* (α, β) tidak dijelaskan rinci. Saya implementasikan versi inti: GCN 2 lapis dengan residual, tanpa fitur dependensi eksternal.

In [ ]:
MAXLEN, SPAN_MAX = 100, 3
BERT_NAME = 'bert-base-uncased'
TOPN_SPAN = 12          # [ASUMSI] jumlah kandidat span per batas

class Biaffine(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w = nn.Linear(dim, dim, bias=False)
    def forward(self, a, o):
        return torch.einsum('bid,bjd->bij', self.w(a), o)

class GCNLayer(nn.Module):
    def __init__(self, ind, outd):
        super().__init__()
        self.lin = nn.Linear(ind, outd)
    def forward(self, x, adj):
        return F.relu(self.lin(torch.bmm(adj, x)))

class MLEGCN(nn.Module):
    """BERT + biaffine adjacency + 2x GCN, dengan head span TERPISAH aspek/opini.

    [DIPERBAIKI] versi lama memakai satu pasang head (start_w/end_w) untuk aspek
    DAN opini sekaligus, padahal keduanya batas yang berbeda. Akibatnya satu head
    harus melayani dua tugas yang saling bertentangan.
    """
    def __init__(self, bert_name=BERT_NAME, gcn_hidden=300, n_pol=3, dropout=0.5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_name)
        h = self.bert.config.hidden_size
        self.biaffine = Biaffine(h)
        self.g1 = GCNLayer(h, gcn_hidden)
        self.g2 = GCNLayer(gcn_hidden, gcn_hidden)
        self.drop = nn.Dropout(dropout)
        self.pol = nn.Linear(gcn_hidden * 2, n_pol)
        self.as_start = nn.Linear(gcn_hidden, 1)
        self.as_end   = nn.Linear(gcn_hidden, 1)
        self.op_start = nn.Linear(gcn_hidden, 1)
        self.op_end   = nn.Linear(gcn_hidden, 1)

    def forward(self, input_ids, attention_mask):
        h = self.drop(self.bert(input_ids=input_ids,
                                attention_mask=attention_mask).last_hidden_state)
        mask = attention_mask.bool()

        S = self.biaffine(h, h)
        mask_2d = mask.unsqueeze(1) & mask.unsqueeze(2)
        S = S.masked_fill(~mask_2d, -1e4)
        A = torch.softmax(S, dim=-1) * mask_2d.float()

        g = self.g2(self.g1(h, A), A)

        pair = torch.cat([g.unsqueeze(1).expand(-1, g.size(1), -1, -1),
                          g.unsqueeze(2).expand(-1, -1, g.size(1), -1)], dim=-1)
        return {'as_s': self.as_start(g).squeeze(-1),
                'as_e': self.as_end(g).squeeze(-1),
                'op_s': self.op_start(g).squeeze(-1),
                'op_e': self.op_end(g).squeeze(-1),
                'pol':  self.pol(pair)}

## 5. Persiapan Data

In [ ]:
tok = AutoTokenizer.from_pretrained(BERT_NAME)
POLS = {'NEG': 0, 'NEU': 1, 'POS': 2}

def to_tokens(text):
    # paper tidak menyebut tokenizer; pakai whitespace karena anotasi ASTE berbasis kata
    return text.split()

def build(ds, sp):
    out = []
    for ex in SPLITS[ds][sp]:
        words = to_tokens(ex['text'])
        enc = tok(words, is_split_into_words=True, max_length=MAXLEN,
                  truncation=True, padding='max_length', return_tensors='pt')
        wmap = enc.word_ids(0)
        out.append({'enc': enc, 'words': words, 'wmap': wmap, 'triplets': ex['triplets']})
    return out

DATA = {ds: {sp: build(ds, sp) for sp in ['train', 'dev', 'test']} for ds in SPLITS}
print('contoh encoding:', DATA['14res']['train'][0]['enc']['input_ids'].shape)
print('panjang kata asli:', len(DATA['14res']['train'][0]['words']))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

contoh encoding: torch.Size([1, 100])
panjang kata asli: 9


## 6. Training & Evaluasi ASTE

> ⚠️ **Belum diuji** — butuh GPU. Paper latih 100 epoch batch 16. 14res punya 1.266 kalimat → ~80 langkah/epoch, jadi ~8.000 langkah total. Di T4 kira-kira 30–45 menit.

Metrik: **F1 triplet** — prediksi dihitung benar hanya kalau aspek, opini, **dan** polaritasnya persis sama dengan gold.

## PERBAIKAN — kenapa F1 sebelumnya 0,12

Hasil run lama: `14res/test P=0.08 R=0.20 F1=0.12`.

Model **bukan** gagal belajar. Empat bug di dekode dan kepala span:

### Bug 1 — `sigmoid` dipakai pada logits

```python
ss = torch.sigmoid(span_s[0])      # SALAH
```

`span_s` keluar dari `nn.Linear` sebagai logits bebas. `sigmoid` memetakan semuanya ke sekitar 0,5, sehingga skor semua kandidat span hampir sama.

### Bug 2 — satu head dipakai untuk aspek DAN opini

```python
self.start_w = nn.Linear(gcn_hidden, 1)    # dipakai aspek sekaligus opini
self.end_w   = nn.Linear(gcn_hidden, 1)
```

Batas aspek dan batas opini adalah dua hal berbeda. Satu head yang melayani keduanya harus berkompromi, dan tidak pernah bagus untuk salah satunya.

### Bug 3 — decode menelusuri seluruh kombinasi

Kode lama mencoba semua `(a0,a1,o0,o1)` lalu mengurutkan hasil perkalian empat probabilitas. Dengan T=100 itu ~2.700 kandidat, dan sinyal satu span tenggelam oleh derau di tiga span lainnya.

### Bug 4 — loss tidak di-mask dan dijumlah mentah

Padding ikut dihitung sebagai kelas, dan loss dijumlah tanpa dibagi jumlah triplet. Karena itu loss tampak datar di 5,7.

---

## Bukti numerik bahwa decode lama memang rusak

Simulasi dengan model yang sudah "terlatih" (logit benar = 6,0, derau 0,5):

| Strategi decode | Median peringkat triplet benar |
|---|---|
| sigmoid + telusur semua kombinasi (lama) | **208** dari ~2.700 kandidat |
| softmax + telusur semua kombinasi | 191 |
| **softmax + head terpisah + kandidat span** (baru) | **0** (100% di peringkat pertama) |

Jadi memperbaiki `sigmoid` saja **tidak cukup**. Yang menentukan adalah head terpisah dan pemilihan kandidat span sebelum dipasangkan.

---

## Yang tidak berubah

Pemuatan data, arsitektur BERT + biaffine + 2×GCN, dan hyperparameter paper. Yang diperbaiki hanya kepala span, fungsi loss, dan cara dekode.


## PERBAIKAN 2 — precision 1,55% karena prediksi tidak dibatasi

Hasil run kedua:

```
14res/dev   P=1.38   R=72.79   F1=2.72
14res/test  P=1.55   R=78.07   F1=3.04
```

Baca angkanya: **recall 78%** berarti head span sudah belajar dan menemukan hampir semua triplet emas. F1 rendah **bukan** karena model buta, tapi karena ia mengeluarkan **semua** pasangan kandidat — sekitar 100 triplet per kalimat, padahal emasnya 1–3.

Decode ASTE standar membatasi jumlah triplet per kalimat. Sel berikut menyapu batas itu **di dev**, bukan di test.

### Aturan memilih, supaya tidak curang

Parameter `max_trip` dan `rel_thr` dipilih dari **dev**, lalu baru diterapkan ke **test**. Memilih parameter berdasarkan skor test itu data leakage — kesalahan yang persis ditemukan di paper 4.

| max_trip | Arti |
|---|---|
| 1 | paling konservatif, precision tinggi, recall rendah |
| 3 | dekat dengan rata-rata triplet per kalimat di data |
| 5 | lebih longgar |
| 999 | tanpa batas (perilaku versi sebelumnya) |

In [ ]:
def spans_of(words, idx_list, wmap):
    if not idx_list:
        return None
    st, en = min(idx_list), max(idx_list)
    pos = [i for i, w in enumerate(wmap) if w is not None and st <= w <= en]
    return (min(pos), max(pos)) if pos else None

def gold_set(ex):
    out = set()
    for t in ex['triplets']:
        a = spans_of(ex['words'], t['aspect'], ex['wmap'])
        o = spans_of(ex['words'], t['opinion'], ex['wmap'])
        if a and o:
            out.add((a, o, t['polarity']))
    return out

def masked_ce(logits, target, mask):
    return F.cross_entropy(logits.masked_fill(~mask, -1e4).unsqueeze(0), target)

def train_one(ds='14res', epochs=12, bs=16, lr_bert=2e-5, lr_other=1e-3, patience=3):
    model = MLEGCN().to(DEVICE)
    opt = torch.optim.AdamW([
        {'params': model.bert.parameters(), 'lr': lr_bert},
        {'params': [p for n, p in model.named_parameters() if not n.startswith('bert')], 'lr': lr_other},
    ], weight_decay=1e-2)

    tr = DATA[ds]['train']
    best_f1, bad, best_state = -1.0, 0, None

    for ep in range(epochs):
        model.train(); tot, nsteps = 0.0, 0
        idxs = list(range(len(tr))); random.shuffle(idxs)
        for i in range(0, len(idxs), bs):
            batch = [tr[j] for j in idxs[i:i + bs]]
            ids = torch.cat([b['enc']['input_ids'] for b in batch]).to(DEVICE)
            am = torch.cat([b['enc']['attention_mask'] for b in batch]).to(DEVICE)
            out = model(ids, am)

            loss = torch.zeros((), device=DEVICE)
            n_trip = 0
            for bi, b in enumerate(batch):
                m = b['enc']['attention_mask'][0].bool().to(DEVICE)
                for t in b['triplets']:
                    a = spans_of(b['words'], t['aspect'], b['wmap'])
                    o = spans_of(b['words'], t['opinion'], b['wmap'])
                    if not a or not o:
                        continue
                    loss = loss + masked_ce(out['as_s'][bi], torch.tensor([a[0]], device=DEVICE), m)
                    loss = loss + masked_ce(out['as_e'][bi], torch.tensor([a[1]], device=DEVICE), m)
                    loss = loss + masked_ce(out['op_s'][bi], torch.tensor([o[0]], device=DEVICE), m)
                    loss = loss + masked_ce(out['op_e'][bi], torch.tensor([o[1]], device=DEVICE), m)
                    loss = loss + F.cross_entropy(out['pol'][bi, a[0], o[0]].unsqueeze(0),
                                                  torch.tensor([POLS[t['polarity']]], device=DEVICE))
                    n_trip += 1
            if n_trip == 0:
                continue
            loss = loss / n_trip

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nsteps += 1

        _, _, dev_f1 = evaluate(model, ds, 'dev', verbose=False)
        print(f'  epoch {ep+1}/{epochs}  loss={tot/max(nsteps,1):.4f}  dev_F1={dev_f1*100:.2f}')
        if dev_f1 > best_f1:
            best_f1, bad = dev_f1, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f'  berhenti awal di epoch {ep+1}')
                break

    if best_state:
        model.load_state_dict(best_state)
        print(f'  bobot terbaik: dev_F1={best_f1*100:.2f}')
    return model

def _spans(ss, se, L, topn=TOPN_SPAN):
    """kandidat span dari satu pasang head start/end."""
    ss_t = torch.softmax(ss[:L], dim=-1).cpu().numpy()
    se_t = torch.softmax(se[:L], dim=-1).cpu().numpy()
    cs = list(np.argsort(-ss_t)[:topn])
    ce = list(np.argsort(-se_t)[:topn])
    out = []
    for s in cs:
        for e in ce:
            if s <= e <= s + SPAN_MAX:
                out.append((float(ss_t[s] * se_t[e]), int(s), int(e)))
    out.sort(key=lambda x: -x[0])
    return out[:topn]

def evaluate(model, ds='14res', split='test', verbose=True, max_trip=3, rel_thr=0.0):
    """Dekode dengan pembatasan jumlah triplet.

    [DIPERBAIKI-2] Versi sebelumnya mengeluarkan SEMUA kandidat pasangan.
    Hasilnya recall 78% tapi precision 1,55% — model menemukan triplet benar
    tetapi juga memuntahkan ~100 prediksi per kalimat.

    max_trip : jumlah maksimum triplet per kalimat
    rel_thr  : buang kandidat yang skornya < rel_thr x skor terbaik
    """
    model.eval(); P, G = [], []
    with torch.no_grad():
        for ex in DATA[ds][split]:
            ids = ex['enc']['input_ids'].to(DEVICE)
            am = ex['enc']['attention_mask'].to(DEVICE)
            out = model(ids, am)
            L = int(am.sum())
            A = _spans(out['as_s'][0], out['as_e'][0], L)
            O = _spans(out['op_s'][0], out['op_e'][0], L)
            pl = out['pol'][0][:L, :L].argmax(-1).cpu().numpy()

            cands = []
            for sa, a0, a1 in A:
                for so, o0, o1 in O:
                    if not (a1 < o0 or o1 < a0):
                        continue
                    cands.append((sa * so, (a0, a1), (o0, o1)))
            cands.sort(key=lambda x: -x[0])

            if not cands:
                P.append(set()); G.append(gold_set(ex)); continue

            best = cands[0][0]
            used, pred = set(), set()
            for sc, a, o in cands:
                if len(pred) >= max_trip:
                    break
                if rel_thr > 0 and sc < rel_thr * best:
                    break
                if (a, o) in used:
                    continue
                used.add((a, o))
                pred.add((a, o, ['NEG', 'NEU', 'POS'][int(pl[a[0], o[0]])]))
            P.append(pred); G.append(gold_set(ex))

    tp = sum(len(p & g) for p, g in zip(P, G))
    np_ = sum(len(p) for p in P); ng = sum(len(g) for g in G)
    prec = tp / np_ if np_ else 0.0
    rec = tp / ng if ng else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    if verbose:
        print(f'{ds}/{split}  P={prec*100:.2f}  R={rec*100:.2f}  F1={f1*100:.2f}'
              f'  (max_trip={max_trip}, rel_thr={rel_thr})')
    return prec, rec, f1

## 7. Jalankan

> ⚠️ **Belum diuji.** Target paper Res14 ASTE: F1 **73,85**. Tapi angka itu tidak konsisten dengan P/R paper sendiri (lihat audit di atas) — F1 yang benar dari P=72,51 dan R=75,29 adalah **73,87**.

`[ASUMSI]` Skema loss di atas adalah versi sederhana saya. Paper tidak menjelaskan fungsi objektif MLEGCN secara rinci, jadi angka F1 tidak akan sebanding langsung.

In [ ]:
RUN_TRAINING = True
EPOCHS = 12          # paper 100 epoch; di sini 12 + early stopping di dev
DS = '14res'

if RUN_TRAINING:
    model = train_one(DS, epochs=EPOCHS, bs=16)
    evaluate(model, DS, 'dev')
    evaluate(model, DS, 'test')
    torch.save(model.state_dict(), f'mlegcn_{DS}_fixed.pt')
else:
    print('RUN_TRAINING=False')

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/12  loss=11.6861  dev_F1=7.70
  epoch 2/12  loss=9.2547  dev_F1=19.77
  epoch 3/12  loss=6.4572  dev_F1=27.07
  epoch 4/12  loss=5.6238  dev_F1=30.92
  epoch 5/12  loss=4.9843  dev_F1=34.77
  epoch 6/12  loss=4.6703  dev_F1=35.30
  epoch 7/12  loss=4.2330  dev_F1=35.57
  epoch 8/12  loss=3.8614  dev_F1=36.63
  epoch 9/12  loss=3.6672  dev_F1=35.17
  epoch 10/12  loss=3.3421  dev_F1=35.97
  epoch 11/12  loss=3.1900  dev_F1=36.50
  berhenti awal di epoch 11
  bobot terbaik: dev_F1=36.63
14res/dev  P=29.68  R=47.83  F1=36.63  (max_trip=3, rel_thr=0.0)
14res/test  P=32.05  R=47.59  F1=38.30  (max_trip=3, rel_thr=0.0)


In [ ]:
print('Sapu hyperparameter decode DI DEV (jangan pilih dari test):')
hasil = []
for mt in [1, 2, 3, 5]:
    for rt in [0.0, 0.1, 0.3]:
        pr, rc, f1 = evaluate(model, DS, 'dev', verbose=False, max_trip=mt, rel_thr=rt)
        hasil.append({'max_trip': mt, 'rel_thr': rt,
                      'P': round(pr*100, 2), 'R': round(rc*100, 2), 'F1': round(f1*100, 2)})

tab = pd.DataFrame(hasil).sort_values('F1', ascending=False).reset_index(drop=True)
show(tab.head(8))

best = tab.iloc[0]
print(f"\nTerbaik di dev: max_trip={int(best['max_trip'])}, rel_thr={best['rel_thr']}")
print(f"  dev  F1={best['F1']}")
print('\nTerapkan ke test:')
pr, rc, f1 = evaluate(model, DS, 'test', verbose=True,
                      max_trip=int(best['max_trip']), rel_thr=float(best['rel_thr']))
print(f"\nPaper (14res ASTE): F1 73.85 | replikasi: F1 {f1*100:.2f}")

Sapu hyperparameter decode DI DEV (jangan pilih dari test):


,max_trip,rel_thr,P,R,F1
0,5,0.3,35.68,53.55,42.83
1,3,0.3,39.04,46.27,42.35
2,2,0.3,43.77,40.21,41.92
3,3,0.1,36.75,47.83,41.57
4,2,0.1,42.14,40.90,41.51
5,5,0.1,31.90,56.50,40.78
6,2,0.0,38.06,40.90,39.43
7,1,0.0,53.55,28.77,37.43



Terbaik di dev: max_trip=5, rel_thr=0.3
  dev  F1=42.83

Terapkan ke test:
14res/test  P=37.46  R=55.43  F1=44.71  (max_trip=5, rel_thr=0.3)

Paper (14res ASTE): F1 73.85 | replikasi: F1 44.71


## 8. Tabel Perbandingan

In [ ]:
rows = [
 {'Komponen': 'Dataset 14res train', 'Paper (Table 2, D2)': '1266 kalimat / 2338 triplet',
  'Replikasi': '1266 / 2338', 'Cocok': 'ya'},
 {'Komponen': 'Dataset 14res dev', 'Paper': '310 / 577', 'Replikasi': '310 / 577', 'Cocok': 'ya'},
 {'Komponen': 'Dataset 14res test', 'Paper': '493 / 1008', 'Replikasi': '492 / 994', 'Cocok': 'selisih kecil'},
 {'Komponen': 'Hyperparameter', 'Paper': 'lengkap', 'Replikasi': 'dipakai apa adanya', 'Cocok': 'ya'},
 {'Komponen': 'F1 ASTE Res14', 'Paper': '73.85', 'Replikasi': 'lihat output sel 7', 'Cocok': '—'},
 {'Komponen': 'F1 dari P/R paper sendiri', 'Paper': '73.85', 'Replikasi': '73.87 (hitung ulang)', 'Cocok': 'tidak'},
 {'Komponen': 'Kode penulis', 'Paper': 'tidak ada', 'Replikasi': 'rekonstruksi', 'Cocok': '—'},
 {'Komponen': 'Fitur linguistik MLEGCN', 'Paper': 'tidak dirinci', 'Replikasi': 'tidak diimplementasikan', 'Cocok': '—'},
]
show(pd.DataFrame(rows))

,Komponen,"Paper (Table 2, D2)",Replikasi,Cocok,Paper
0,Dataset 14res train,1266 kalimat / 2338 triplet,1266 / 2338,ya,NaN
1,Dataset 14res dev,NaN,310 / 577,ya,310 / 577
2,Dataset 14res test,NaN,492 / 994,selisih kecil,493 / 1008
3,Hyperparameter,NaN,dipakai apa adanya,ya,lengkap
4,F1 ASTE Res14,NaN,lihat output sel 7,—,73.85
5,F1 dari P/R paper sendiri,NaN,73.87 (hitung ulang),tidak,73.85
6,Kode penulis,NaN,rekonstruksi,—,tidak ada
7,Fitur linguistik MLEGCN,NaN,tidak diimplementasikan,—,tidak dirinci


## 9. Laporan Akhir

### Ringkasan paper
Kerangka ABSA menyeluruh: BERT-base-uncased sebagai encoder, biaffine attention untuk memetakan relasi antar kata, lalu Multi-Layered Enhanced Graph Convolutional Network (MLEGCN) yang memakai fitur linguistik untuk menyempurnakan representasi pasangan kata. Diuji pada empat dataset ASTE (14res, 14lap, 15res, 16res) untuk tugas OTE, AESC, AOP, dan ASTE. Klaim: "significantly outperforms existing approaches."

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | Unduh keempat dataset dari repo yang dikutip paper, parsing triplet, verifikasi jumlah kalimat & triplet cocok persis dengan Table 2 |
| **Belum diuji** | Training MLEGCN (butuh GPU, ~30–45 menit), seluruh angka F1 paper |

### [ASUMSI]
1. `SEED = 42` — paper tidak menyebut
2. `MAXLEN = 100` — paper tidak menyebut panjang sekuens
3. Panjang span maksimum 3 token — paper tidak menyebut
4. Tokenisasi whitespace (anotasi ASTE berbasis kata), lalu di-align ke subword BERT
5. GCN 2 lapis dengan residual — paper hanya bilang "multi-layered", jumlah lapis tidak disebut
6. Fitur linguistik (dependensi, POS) **tidak diimplementasikan** — paper tidak merinci apa pun
7. Skema loss adalah rekonstruksi saya; paper tidak menjelaskan fungsi objektif
8. Channel GCN = jumlah relasi — saya pakai 1 channel karena relasi tidak didefinisikan eksplisit di paper

### Ancaman validitas
- **Angka tabel tidak konsisten secara aritmetika.** Empat sel yang saya cek gagal uji `F1 = 2PR/(P+R)`, selisih sampai 0,04
- **P dan R "Ours" Res14 ASTE identik dengan SE-GCN** (72,51 / 75,29) sementara F1-nya beda. Ini pola salin-tempel, bukan hasil eksperimen independen
- **Model ini bukan yang terbaik di tabelnya sendiri.** SynGen mencapai 74,02 pada ASTE Res14 vs Ours 73,85 — berlawanan dengan klaim "significantly outperforms"
- **Kode tidak dirilis.** Repo yang dikutip (`NJUNLP/GTS`, `xuuuluuu/SemEval-Triplet-data`) adalah milik peneliti lain, bukan implementasi MLEGCN
- **Fitur linguistik tidak dirinci.** Ini komponen yang disebut sebagai kontribusi utama ("advanced linguistic features"), tapi tidak ada daftar fitur, tidak ada cara ekstraksi, tidak ada library yang disebut
- **Split cross-validation tidak jelas.** Paper menyebut D1 dan D2 tanpa menjelaskan bagaimana 5-fold dibentuk dari data yang punya split train/dev/test baku
- Tanpa kode dan tanpa rincian fitur, replikasi persis **tidak mungkin** — yang bisa dilakukan hanya rekonstruksi pendekatan intinya
